# 🦙 On-Device Hybrid RAG with Meta Llama 3.2 & SQLite FTS5 (BM25 + RRF)

> **Author:** Çağrı Giray Keşan ([@Cagrik34](https://github.com/Cagrik34))  
> **Focus:** Llama 3.2 (1B/3B On-Device), SQLite FTS5 BM25, Dense Embeddings, Reciprocal Rank Fusion (RRF), Strict In-Text Citations

---

## 📌 1. Overview
Deploying generative AI on edge devices requires minimal compute footprints and zero reliance on heavy cloud vector databases. 

This recipe provides an end-to-end guide on building a **zero-cloud, on-device Hybrid RAG engine** for **Meta Llama 3.2 (1B / 3B)** using **SQLite FTS5 BM25** and Dense Vector Cosine Similarity fused via **Reciprocal Rank Fusion (RRF, $k=60$)**.

In [ ]:
%pip install -q numpy

import sys
import sqlite3
import numpy as np
from typing import List, Tuple, Dict, Any

print("✅ Environment initialized.")

## 🏗️ 2. SQLite Dual-Table Hybrid Engine

In [ ]:
class LlamaHybridStore:
    def __init__(self, db_path: str = ":memory:"):
        self.conn = sqlite3.connect(db_path)
        self._init_schema()

    def _init_schema(self) -> None:
        with self.conn:
            self.conn.execute("""
                CREATE TABLE IF NOT EXISTS documents (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    source TEXT NOT NULL,
                    content TEXT NOT NULL,
                    embedding BLOB NOT NULL
                )
            """)
            self.conn.execute("""
                CREATE VIRTUAL TABLE IF NOT EXISTS documents_fts USING fts5(
                    content,
                    source UNINDEXED,
                    tokenize='unicode61'
                )
            """)

    def insert(self, source: str, content: str, embedding: List[float]) -> None:
        vec = np.array(embedding, dtype=np.float32)
        norm = np.linalg.norm(vec)
        if norm > 0:
            vec = vec / norm
        with self.conn:
            self.conn.execute(
                "INSERT INTO documents (source, content, embedding) VALUES (?, ?, ?)",
                (source, content, vec.tobytes())
            )
            self.conn.execute(
                "INSERT INTO documents_fts (content, source) VALUES (?, ?)",
                (content, source)
            )

    def hybrid_search(self, query_text: str, query_vec: List[float], top_k: int = 2, rrf_k: int = 60) -> List[Dict[str, Any]]:
        q_arr = np.array(query_vec, dtype=np.float32)
        q_norm = np.linalg.norm(q_arr)
        if q_norm > 0:
            q_arr = q_arr / q_norm

        # Dense Search
        cursor = self.conn.execute("SELECT id, source, content, embedding FROM documents")
        dense_hits = []
        for doc_id, src, content, blob in cursor.fetchall():
            doc_vec = np.frombuffer(blob, dtype=np.float32)
            sim = float(np.dot(q_arr, doc_vec))
            dense_hits.append((doc_id, src, content, sim))
        dense_hits.sort(key=lambda x: x[3], reverse=True)

        # Sparse BM25 Search
        clean_tokens = [t for t in query_text.replace("'", "").replace('"', '').split() if len(t) > 1]
        fts_query = " OR ".join(f'"{t}"' for t in clean_tokens) if clean_tokens else ""
        sparse_hits = []
        if fts_query:
            cursor = self.conn.execute(
                "SELECT rowid, source, content, rank FROM documents_fts WHERE documents_fts MATCH ? ORDER BY rank LIMIT 10",
                (fts_query,)
            )
            for doc_id, src, content, rank in cursor.fetchall():
                sparse_hits.append((doc_id, src, content, 1.0 / (1.0 + abs(float(rank)))))

        # Reciprocal Rank Fusion (RRF k=60)
        fused = {}
        meta_map = {}
        for rank, (doc_id, src, content, sim) in enumerate(dense_hits[:10], start=1):
            key = f"{src}::{content[:50]}"
            meta_map[key] = (src, content, "dense")
            fused[key] = fused.get(key, 0.0) + (1.0 / (rrf_k + rank))

        for rank, (doc_id, src, content, bm25) in enumerate(sparse_hits, start=1):
            key = f"{src}::{content[:50]}"
            if key not in meta_map:
                meta_map[key] = (src, content, "bm25")
            else:
                meta_map[key] = (src, content, "hybrid")
            fused[key] = fused.get(key, 0.0) + (1.0 / (rrf_k + rank))

        sorted_keys = sorted(fused.keys(), key=lambda k: fused[k], reverse=True)[:top_k]
        results = []
        for idx, key in enumerate(sorted_keys, start=1):
            src, content, match_type = meta_map[key]
            results.append({
                "citation_index": idx,
                "source": src,
                "content": content,
                "rrf_score": round(fused[key], 4),
                "match_type": match_type
            })
        return results

print("✅ LlamaHybridStore compiled successfully.")

## 📄 3. Knowledge Base Ingestion

In [ ]:
store = LlamaHybridStore()
docs = [
    ("q3_financial_report.pdf", "CodePulse engineering project total Q3 budget was allocated at 2,340,000 TL with 15 active developers.", [0.8, 0.1, 0.2] + [0.0] * 381),
    ("architecture_specs.md", "Zenith AI leverages Microsoft phi-4-mini (3.8B parameters) for local zero-cloud inference.", [0.2, 0.9, 0.1] + [0.0] * 381),
    ("hr_policy_2026.docx", "Remote work expense allowance is capped at 15,000 TL per employee quarterly.", [0.1, 0.1, 0.8] + [0.0] * 381)
]
for src, text, emb in docs:
    store.insert(src, text, emb)
print("✅ Knowledge base seeded.")

## 🤖 4. Generating Llama 3.2 Formatted Grounded Prompts

In [ ]:
query = "What is the allocated budget for the CodePulse project?"
dummy_q_vec = [0.75, 0.15, 0.25] + [0.0] * 381
hits = store.hybrid_search(query, dummy_q_vec, top_k=2)

context_str = "\n\n".join([
    f"[{c['citation_index']}] (Source: {c['source']}) {c['content']}"
    for c in hits
])

llama3_prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert AI assistant powered by Meta Llama 3.2. Use the provided context to answer the user inquiry. For every claim or numerical figure you cite, you MUST include the reference citation index in brackets like [1] or [2].<|eot_id|><|start_header_id|>user<|end_header_id|>

Context Information:
---------------------
{context_str}
---------------------

Question: {query}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
"""

print("📋 Formatted Prompt for Llama 3.2 Inference:")
print(llama3_prompt)